# Open API를 활용한 네이버 뉴스 검색

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경변수 관리
- 등록된 애플리케이션 페이지에서 제공되는 Client ID, Secret은 절대 외부로 노출되면 안 된다
- dotenv (.env)를 통해서 관리
    - `pip install python-dotenv` 패키지 설치
    - .gitignore 파일에 .env 무시하는 구문 추가

In [1]:
!pip install python-dotenv

### 프로젝트 히위 폴더에 .env 파일을 만들어 아래 값 입력
```
NAVER_CLIENT_ID={{Client Id}}
NAVER_CLIENT_SECRET={{Client Secret}}
```
- 네이버 개발자센터에 등록된 애플리케이션에서 확인 가능

In [2]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv

load_dotenv() # .env 파일을 읽어와 자동으로 환경 변수로 등록
# 읽어오기 성공 시 True, 실패 시 False

True

In [3]:
# 환경 변수에서 .env 등록한 내용 얻어오기
import os

NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

# api key, 비밀번호를 print 하는 구문을 실수로 남겨놓으면 노출될 가능성 있음
# -> jupyter 변수 탭에서 확인

# .env 내용이 환경변수로 등록되지 않은 경우
if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('네이버 클라이언트 아이디 또는 시크릿이 환경 변수에 등록되지 않았습니다.')

## 3. API 요청
- 파이썬에서 웹 요청(http)을 처리하기 위해서는 `requests` 라이브러리 필요

!pip install requests

In [4]:
!pip install requests

In [5]:
# 네이버 뉴스 검색 API
import urllib.request
import socket

encText = urllib.parse.quote('인공지능') # url encoding 작업
# 한글 -> %시작하는 문자로 변경

url = f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'
# ? 이후 -> 쿼리스트링 == 전달할 값(파라미터)가 작성된 문자열

# 요청 객체 생성
request = urllib.request.Request(url)

# API 인증 정보를 요청 해더에 추가
request.add_header("X-Naver-Client-Id", NAVER_CLIENT_ID)
request.add_header("X-Naver-Client-Secret", NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        # 지정된 주소로 요청 -> 결과를 response로 전달 받음
        # 단, 요청 대기 시간이 10초를 초과하면 중지

        # HTTP 응답 상태 코드, 200이면 정상 응답 == 성공
        response_code = response.getcode()

        # 응답 본문 확인 (bytes -> UTF-8로 변환)
        response_body = response.read().decode('utf-8')

        print("response_code: ", response_code)
        print(response_body)
        # 응답 본문이 JSON(str 타입) 형태
        # -> 이용이 필요할 경우 parsing 작업 필수

except socket.timeout:
    print("요청 시간 10초 초과")

response_code:  200
{
	"lastBuildDate":"Mon, 15 Jun 2026 15:11:54 +0900",
	"total":4046595,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"현대차 노조, 중노위에 노동쟁의 조정 신청 “24일 파업 찬반투표”",
			"originallink":"https:\/\/www.etoday.co.kr\/news\/view\/2593720",
			"link":"https:\/\/www.etoday.co.kr\/news\/view\/2593720",
			"description":"노조는 올해 교섭에서 월 기본급 14만9600원(호봉승급분 제외) 인상과 전년도 순이익의 30% 성과급 지급, <b>인공지능<\/b>(AI) 도입에 따른 고용 및 노동조건 보장 등을 요구하고 있다. 이와 함께 완전월급제 시행, 상여금 750%에서... ",
			"pubDate":"Mon, 15 Jun 2026 15:10:00 +0900"
		},
		{
			"title":"민형배 전남광주통합특별시 당선인, 시민과 직접 소통 나선다",
			"originallink":"https:\/\/news.tf.co.kr\/read\/national\/2332834.htm",
			"link":"https:\/\/n.news.naver.com\/mnews\/article\/629\/0000507696?sid=102",
			"description":"타운홀미팅에서 민 당선인은 전남광주지역 발전을 위해 구상하고 있는 '<b>인공지능<\/b>(AI) 중심의 창업도시'에 대해 설명하고 관련 기관의 의견을 청취한다. 또 여성·노동·문화 등의 주제를 놓고 관련 단체의 의견을 수렴하고... ",
			"pubDate":"Mon, 15 Jun 2026 15:10:00 +0900"
		},
		{
			"title":"양극재는 헝가리, 니켈은 인니…에코프로, 공급망 완성 '속도'",
			"ori

In [6]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : "인공지능",
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(url encoding도 같이 실행됨)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    print("response_code: ", response_code)
    pprint(data)

    # pprint(data['items'][0]) # 첫 번째 기사

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
{'display': 10,
 'items': [{'description': '노조는 올해 교섭에서 월 기본급 14만9600원(호봉승급분 제외) 인상과 전년도 순이익의 '
                           '30% 성과급 지급, <b>인공지능</b>(AI) 도입에 따른 고용 및 노동조건 보장 등을 '
                           '요구하고 있다. 이와 함께 완전월급제 시행, 상여금 750%에서... ',
            'link': 'https://www.etoday.co.kr/news/view/2593720',
            'originallink': 'https://www.etoday.co.kr/news/view/2593720',
            'pubDate': 'Mon, 15 Jun 2026 15:10:00 +0900',
            'title': '현대차 노조, 중노위에 노동쟁의 조정 신청 “24일 파업 찬반투표”'},
           {'description': '타운홀미팅에서 민 당선인은 전남광주지역 발전을 위해 구상하고 있는 '
                           "'<b>인공지능</b>(AI) 중심의 창업도시'에 대해 설명하고 관련 기관의 의견을 "
                           '청취한다. 또 여성·노동·문화 등의 주제를 놓고 관련 단체의 의견을 수렴하고... ',
            'link': 'https://n.news.naver.com/mnews/article/629/0000507696?sid=102',
            'originallink': 'https://news.tf.co.kr/read/national/2332834.htm',
            'pubDate': 'Mon, 15 Jun 2026 15:10:00 +0900',
            'title': '민형배 전남광

In [7]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.xml'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : "인공지능",
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(url encoding도 같이 실행됨)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.content # 응답 문자열(xml)

    # XML 해석 파이썬 기본 내장 라이브러리
    # -> xml.etree.ElementTree

    # XML 파일로 저장
    with open('response.xml', 'wb') as f:
        f.write(data)

    print("response.xml 저장 완료")

except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")

response.xml 저장 완료
